In [1]:
%matplotlib inline
import json

import pandas as pd

from src.config import CUTOFF_STEP, DATA_PATH, FEATURE_LIST
from src.evaluate import evaluate_test
from src.feature_engineering import build_features, temporal_split
from src.model import train_classifier

df = pd.read_parquet(DATA_PATH)
print(df.shape, CUTOFF_STEP)

(6362620, 12) 594


In [2]:
X_all = build_features(df)
print(X_all.shape)
X_all.head()

(6362620, 10)


,hour_of_day,amount,oldbalanceOrg,oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,orig_drain_ratio
0,1,9839.64,170136.0,0.0,0,0,0,1,0,0.057834
1,1,1864.28,21249.0,0.0,0,0,0,1,0,0.087735
2,1,181.00,181.0,0.0,0,0,0,0,1,1.000000
3,1,181.00,181.0,21182.0,0,1,0,0,0,1.000000
4,1,11668.14,41554.0,0.0,0,0,0,1,0,0.280795


In [3]:
train, test = temporal_split(df)
print(len(train), len(test), train["step"].max(), test["step"].min())

6239040 123580 594 595


In [4]:
model = train_classifier(build_features(train), train["isFraud"])
score = model.predict_proba(build_features(test))[:, 1]
print(len(FEATURE_LIST), score.shape)

10 (123580,)


In [5]:
metrics = evaluate_test(test, score)
summary = pd.DataFrame(
    {
        name: {"rows": m["n_rows"], "fraud": m["n_fraud"], "prevalence": m["prevalence"], "pr_auc": m["pr_auc"]}
        for name, m in metrics.items()
    }
).T
summary

,rows,fraud,prevalence,pr_auc
transfer_cash_out,51286.0,1654.0,0.032251,0.980905
all_rows,123580.0,1654.0,0.013384,0.980701


In [6]:
pd.DataFrame(
    {name: {"P@" + k: v for k, v in m["precision_at_k"].items()} for name, m in metrics.items()}
)

,transfer_cash_out,all_rows
P@51,1.000000,NaN
P@100,1.000000,1.000000
P@256,1.000000,NaN
P@500,1.000000,1.000000
P@1000,0.998000,0.998000
P@1654,0.960097,0.960097
P@124,NaN,1.000000
P@618,NaN,1.000000


In [7]:
pd.DataFrame(
    {name: {"R@" + k: v for k, v in m["recall_at_k"].items()} for name, m in metrics.items()}
)

,transfer_cash_out,all_rows
R@51,0.030834,NaN
R@100,0.060459,0.060459
R@256,0.154776,NaN
R@500,0.302297,0.302297
R@1000,0.603386,0.603386
R@1654,0.960097,0.960097
R@124,NaN,0.074970
R@618,NaN,0.373640


In [8]:
rows = []
for name, m in metrics.items():
    for target, t in m["at_target_recall"].items():
        rows.append({"population": name, "target_recall": target, "fpr": t["fpr"], "fp": t["fp"], "flagged": t["flagged"]})
pd.DataFrame(rows)

,population,target_recall,fpr,fp,flagged
0,transfer_cash_out,0.8,0.000081,4,1328
1,transfer_cash_out,0.9,0.000685,34,1523
2,all_rows,0.8,0.000033,4,1328
3,all_rows,0.9,0.000279,34,1523


In [9]:
saved = json.loads((DATA_PATH.parents[2] / "reports" / "metrics.json").read_text())
print({k: round(v["pr_auc"], 6) for k, v in saved["test"].items()})
print({k: round(v["pr_auc"], 6) for k, v in metrics.items()})

{'transfer_cash_out': 0.980905, 'all_rows': 0.980701}
{'transfer_cash_out': 0.980905, 'all_rows': 0.980701}


In [10]:
ablation = json.loads((DATA_PATH.parents[2] / "reports" / "ablation.json").read_text())
pd.DataFrame({k: v["pr_auc"] for k, v in ablation.items()}).T

,transfer_cash_out,all_rows
"1 amount, hour",0.403479,0.371453
2 + type,0.503603,0.503603
3 + old balances,0.976813,0.976813
4 + drain ratio,0.980905,0.980701
5 + new balances,0.997242,0.997220
